# Mundial 2026 Predictor — Notebook de Validación del Modelo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angel011298/mundial-2026-predictor/blob/master/notebooks/mundial_modelo.ipynb)

Este notebook **espeja en Python** la metodología de `src/model/dixonColes.js` y `src/model/monteCarlo.js` con el propósito exclusivo de **validación matemática**. No se despliega.

## Objetivos
1. Descargar datos históricos de partidos internacionales (martj42 dataset)
2. Estimar los parámetros de ataque/defensa por equipo
3. Implementar el mismo modelo Dixon-Coles + tilt Elo en Python
4. Generar el heatmap, barras 1X2 y Top-10 marcadores
5. Comparar numéricamente con la salida JSON de `scripts/exportModelData.mjs`

## Partido de referencia: **ARG vs FRA** (Argentina como local hipotético)

---
> **Nota**: Los valores de `avgGF`, `avgGA`, `form` y `eloRating` usados para la
> comparación definitiva provienen de `src/data/worldcup2026.json` y
> `src/data/team-crosswalk.json` — los mismos que usa la app en producción.


In [ ]:
# Instalar dependencias (solo Colab / entorno limpio)
!pip install pandas numpy scipy matplotlib seaborn requests -q

In [ ]:
import json, math, requests
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from   scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#09090b',
    'axes.facecolor':   '#18181b',
    'axes.edgecolor':   '#27272a',
    'axes.labelcolor':  '#a1a1aa',
    'text.color':       '#e4e4e7',
    'xtick.color':      '#71717a',
    'ytick.color':      '#71717a',
    'grid.color':       '#27272a',
    'font.family':      'DejaVu Sans',
})

print('Librerías cargadas.')

## 1 — Constantes (idénticas a los módulos JS)

| Constante | JS | Python | Significado |
|-----------|-----|--------|-------------|
| `LEAGUE_AVG` | `1.32` | `1.32` | Media de goles por equipo/partido en Mundiales |
| `GAMMA` | `0.15` | `0.15` | Peso del ajuste Elo sobre λ |
| `ELO_DIVISOR` | `400` | `400` | Divisor estándar del sistema Elo |
| `HOST_ADV_FACTOR` | `1.12` | `1.12` | Ventaja de goles para sedes (USA/CAN/MEX) |
| `HOME_ADV_ELO` | `100` | `100` | Puntos Elo equivalentes a jugar en casa |


In [ ]:
LEAGUE_AVG      = 1.32
GAMMA           = 0.15
ELO_DIVISOR     = 400
HOST_ADV_FACTOR = 1.12
HOME_ADV_ELO    = 100
HOST_CODES      = {'USA', 'CAN', 'MEX'}
MAX_GOALS       = 8    # techo de la matriz
FIXED_SEED      = 42

print('Constantes definidas.')

## 2 — Funciones auxiliares (port directo de JS)


In [ ]:
def form_multiplier(form: str = '') -> float:
    """Ajuste de lambda por forma reciente (últimos 5 partidos).
    Puerto exacto de formMultiplier() en dixonColes.js
    Retorna: [0.82, 1.18]
    """
    chars = list(form.upper())[-5:]
    if not chars:
        return 1.0
    score = sum(1 if c == 'W' else 0 if c == 'D' else -1.2 for c in chars)
    return max(0.82, min(1.18, 1 + (score / 5) * 0.18))


# Verificación rápida
assert abs(form_multiplier('WWWWW') - 1.18) < 1e-10, 'WWWWW debe dar 1.18'
assert abs(form_multiplier('LLLLL') - 0.82) < 1e-10, 'LLLLL debe dar 0.82'
assert abs(form_multiplier('WWWDW') - (1 + (4/5)*0.18)) < 1e-9, 'WWWDW falla'
print('form_multiplier: OK')


# --- Modelo Analítico: Dixon-Coles (SIN Elo) ---
def analytical_lambdas(home: dict, away: dict) -> tuple[float, float]:
    """Calcula (lambda_home, lambda_away) con el modelo DC puro.
    Puerto de dixonColesProbs() en dixonColes.js (sin ajuste Elo).
    """
    host_adv = HOST_ADV_FACTOR if home['code'] in HOST_CODES else 1.0
    fm_h = form_multiplier(home.get('form', ''))
    fm_a = form_multiplier(away.get('form', ''))

    alpha_h = home['avgGF'] / LEAGUE_AVG  # ataque relativo
    beta_a  = away['avgGA'] / LEAGUE_AVG  # vulnerabilidad defensiva del visitante
    alpha_a = away['avgGF'] / LEAGUE_AVG
    beta_h  = home['avgGA'] / LEAGUE_AVG

    lam_h = alpha_h * beta_a * LEAGUE_AVG * host_adv * fm_h
    lam_a = alpha_a * beta_h * LEAGUE_AVG              * fm_a
    return min(lam_h, 6.0), min(lam_a, 6.0)


# --- Modelo MC: DC + tilt Elo (MISMO que deriveLambdas en monteCarlo.js) ---
def derive_lambdas(home: dict, away: dict) -> tuple[float, float]:
    """DC base + multiplicador Elo. Puerto EXACTO de deriveLambdas()."""
    host_adv = HOST_ADV_FACTOR if home['code'] in HOST_CODES else 1.0
    fm_h = form_multiplier(home.get('form', ''))
    fm_a = form_multiplier(away.get('form', ''))

    lam_h = home['attack']  * away['defense'] * LEAGUE_AVG * host_adv * fm_h
    lam_a = away['attack']  * home['defense'] * LEAGUE_AVG              * fm_a

    elo_adv  = HOME_ADV_ELO if home['code'] in HOST_CODES else 0
    elo_diff = (home['elo'] - away['elo']) + elo_adv
    tilt     = math.exp(GAMMA * elo_diff / ELO_DIVISOR)
    lam_h   *= math.sqrt(tilt)
    lam_a   /= math.sqrt(tilt)

    return max(0.15, min(lam_h, 6.0)), max(0.15, min(lam_a, 6.0))


print('derive_lambdas, analytical_lambdas: OK')

## 3 — Datos históricos (martj42 dataset)

Descargamos el CSV del repositorio [martj42/international_results](https://github.com/martj42/international_results)
y lo usamos para mostrar cómo se estimarían `avgGF` y `avgGA` en un pipeline real.
Los valores definitivos para la comparación vienen de `worldcup2026.json`.


In [ ]:
CSV_URL = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'

print('Descargando dataset internacional…')
df = pd.read_csv(CSV_URL, parse_dates=['date'])
print(f'Total de partidos: {len(df):,} (hasta {df.date.max().date()})')
df.tail(3)

In [ ]:
# Filtrar: Copa del Mundo (oficial FIFA) desde 2010 para calcular stats de equipo
WC = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'] >= '2010-01-01')
].copy()

# También incluir eliminatorias recientes para más datos
RECENT = df[
    (df['date'] >= '2022-01-01') &
    (df['tournament'].isin(['FIFA World Cup', 'FIFA World Cup qualification',
                            'Friendly', 'Copa América', 'UEFA Euro']))
].copy()

def team_stats(name: str, data: pd.DataFrame) -> dict:
    """avgGF y avgGA para un equipo en un dataset dado."""
    home_games = data[data.home_team == name]
    away_games = data[data.away_team == name]
    goals_for  = pd.concat([
        home_games.home_score,
        away_games.away_score
    ])
    goals_ag   = pd.concat([
        home_games.away_score,
        away_games.home_score
    ])
    n = len(goals_for)
    if n == 0:
        return {'avgGF': 1.32, 'avgGA': 1.32, 'games': 0}
    return {
        'avgGF': round(goals_for.mean(), 4),
        'avgGA': round(goals_ag.mean(),  4),
        'games': n,
    }

# Estadísticas ARG y FRA en partidos recientes (post-2022)
arg_stats = team_stats('Argentina', RECENT)
fra_stats = team_stats('France',    RECENT)

print('Argentina (post-2022):', arg_stats)
print('France    (post-2022):', fra_stats)
print()
print('Comparar con worldcup2026.json:')
print('  Argentina: avgGF=2.3, avgGA=0.7')
print('  France:    avgGF=2.2, avgGA=0.8')

## 4 — Parámetros del partido (desde worldcup2026.json)

Usamos los valores exactos de la app para que la comparación con el JSON sea justa.


In [ ]:
# Valores de worldcup2026.json y team-crosswalk.json (mismos que usa la app)
ARG = {
    'code':    'ARG',
    'name':    'Argentina',
    'avgGF':   2.3,
    'avgGA':   0.7,
    'form':    'WWWDW',
    'elo':     2076,
    'attack':  2.3 / LEAGUE_AVG,
    'defense': 0.7 / LEAGUE_AVG,
}

FRA = {
    'code':    'FRA',
    'name':    'France',
    'avgGF':   2.2,
    'avgGA':   0.8,
    'form':    'WWDWW',
    'elo':     2003,
    'attack':  2.2 / LEAGUE_AVG,
    'defense': 0.8 / LEAGUE_AVG,
}

print(f'ARG  attack={ARG["attack"]:.4f}  defense={ARG["defense"]:.4f}  elo={ARG["elo"]}')
print(f'FRA  attack={FRA["attack"]:.4f}  defense={FRA["defense"]:.4f}  elo={FRA["elo"]}')
print(f'form_multiplier(ARG)={form_multiplier(ARG["form"]):.4f}')
print(f'form_multiplier(FRA)={form_multiplier(FRA["form"]):.4f}')

## 5 — Cálculo de λ y comparación con JS


In [ ]:
# Cargar salida del script Node.js (debe existir: node scripts/exportModelData.mjs)
import os
JS_OUTPUT_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.', 'model_output.json')
# En Colab, el archivo estará en el mismo directorio que el notebook
try:
    with open(JS_OUTPUT_PATH) as f:
        JS = json.load(f)
    print(f'Cargado: {JS_OUTPUT_PATH}')
    print(f'Partido: {JS["match"]["home"]} vs {JS["match"]["away"]}')
    JS_LOADED = True
except FileNotFoundError:
    print('model_output.json no encontrado. Generarlo con:')
    print('  node scripts/exportModelData.mjs')
    print('Continuando sin comparación JS…')
    JS = None
    JS_LOADED = False

In [ ]:
# ── Lambdas analíticas (DC sin Elo) ──────────────────────────────────────────
lam_H_dc, lam_A_dc = analytical_lambdas(ARG, FRA)

print('=== Modelo Analítico (Dixon-Coles, sin Elo) ===')
print(f'  λ_ARG (home) = {lam_H_dc:.6f}')
print(f'  λ_FRA (away) = {lam_A_dc:.6f}')

if JS_LOADED:
    js_lh = JS['analytical']['lambdaH']
    js_la = JS['analytical']['lambdaA']
    print(f'  JS:  λH={js_lh}  λA={js_la}')
    assert abs(lam_H_dc - js_lh) < 1e-4, f'λH analítico no coincide: {lam_H_dc} vs {js_lh}'
    assert abs(lam_A_dc - js_la) < 1e-4, f'λA analítico no coincide: {lam_A_dc} vs {js_la}'
    print('  ✓ Valores analíticos coinciden con JS (tolerancia 1e-4)')

print()

# ── Lambdas MC (DC + tilt Elo) ────────────────────────────────────────────────
lam_H_mc, lam_A_mc = derive_lambdas(ARG, FRA)

elo_diff = ARG['elo'] - FRA['elo']
tilt     = math.exp(GAMMA * elo_diff / ELO_DIVISOR)

print('=== Modelo MC (DC + tilt Elo) ===')
print(f'  ΔElo = {elo_diff}  →  tilt = exp(0.15×{elo_diff}/400) = {tilt:.6f}')
print(f'  √tilt = {math.sqrt(tilt):.6f}')
print(f'  λ_ARG = {lam_H_dc:.6f} × √tilt = {lam_H_mc:.6f}')
print(f'  λ_FRA = {lam_A_dc:.6f} / √tilt = {lam_A_mc:.6f}')

if JS_LOADED:
    js_lh_mc = JS['monteCarlo']['lambdaH']
    js_la_mc = JS['monteCarlo']['lambdaA']
    print(f'  JS:  λH={js_lh_mc}  λA={js_la_mc}')
    assert abs(lam_H_mc - js_lh_mc) < 1e-4, f'λH MC no coincide: {lam_H_mc} vs {js_lh_mc}'
    assert abs(lam_A_mc - js_la_mc) < 1e-4, f'λA MC no coincide: {lam_A_mc} vs {js_la_mc}'
    print('  ✓ Valores MC coinciden con JS (tolerancia 1e-4)')

## 6 — Matriz de probabilidades (Poisson bivariado)

`scipy.stats.poisson.pmf(k, λ)` es el análogo directo de `poisson(lambda, k)` en `dixonColes.js`.

```python
# Python
P(h, a) = scipy.stats.poisson.pmf(h, λ_home) * scipy.stats.poisson.pmf(a, λ_away)
```
```js
// JavaScript (dixonColes.js)
P(h, a) = poisson(lambdaHome, h) * poisson(lambdaAway, a)
```


In [ ]:
def poisson_matrix(lam_h: float, lam_a: float, n: int = MAX_GOALS) -> np.ndarray:
    """Matriz n×n de probabilidades de marcadores.
    mat[h][a] = P(goals_home=h) * P(goals_away=a)
    Análoga a scoreMatrix() en dixonColes.js.
    """
    ph = stats.poisson.pmf(np.arange(n), lam_h)  # shape (n,)
    pa = stats.poisson.pmf(np.arange(n), lam_a)
    return np.outer(ph, pa)   # mat[h, a]


mat_dc = poisson_matrix(lam_H_dc, lam_A_dc)   # analítico (8×8)

# Probabilidades 1X2
p_home_dc = float(np.sum([mat_dc[h, a] for h in range(MAX_GOALS) for a in range(MAX_GOALS) if h > a]))
p_draw_dc = float(np.sum([mat_dc[h, a] for h in range(MAX_GOALS) for a in range(MAX_GOALS) if h == a]))
p_away_dc = float(np.sum([mat_dc[h, a] for h in range(MAX_GOALS) for a in range(MAX_GOALS) if h < a]))
total     = p_home_dc + p_draw_dc + p_away_dc
p_home_dc /= total; p_draw_dc /= total; p_away_dc /= total

print(f'Analítico 1X2: {p_home_dc*100:.1f}% / {p_draw_dc*100:.1f}% / {p_away_dc*100:.1f}%')

if JS_LOADED:
    js_ph = JS['analytical']['pHome']
    js_pd = JS['analytical']['pDraw']
    js_pa = JS['analytical']['pAway']
    print(f'JS           : {js_ph*100:.1f}% / {js_pd*100:.1f}% / {js_pa*100:.1f}%')
    assert abs(p_home_dc - js_ph) < 5e-4, f'pHome analítico difiere: {p_home_dc:.6f} vs {js_ph}'
    assert abs(p_draw_dc - js_pd) < 5e-4
    assert abs(p_away_dc - js_pa) < 5e-4
    print('  ✓ P(1/X/2) analítico coincide con JS')

# También verificar la matriz celda a celda (6×6)
if JS_LOADED:
    js_mat6 = np.array(JS['analytical']['matrix6x6'])
    py_mat6 = mat_dc[:6, :6]
    max_diff = np.max(np.abs(py_mat6 - js_mat6))
    print(f'  Max diff celda 6×6: {max_diff:.2e} (tolerancia 1e-4)')
    assert max_diff < 1e-4, f'Matriz 6×6 difiere demasiado: {max_diff}'
    print('  ✓ Matriz 6×6 celda a celda dentro de tolerancia')

In [ ]:
# ── Heatmap — réplica del ScorelineHeatmap.jsx ───────────────────────────────

# Paleta zinc-900 → emerald (#18181b → #10b981)
ZINC900  = np.array([24, 24, 27]) / 255
EMERALD  = np.array([16, 185, 129]) / 255
cmap_em  = mcolors.LinearSegmentedColormap.from_list('zinc_emerald', [ZINC900, EMERALD])

N_DISP = 6
mat6   = mat_dc[:N_DISP, :N_DISP]

fig, ax = plt.subplots(figsize=(7, 6))
ax.set_facecolor('#09090b')
fig.patch.set_facecolor('#09090b')

# Normalizar a la raíz cuadrada (igual que la app)
disp = np.sqrt(mat6 / mat6.max())
im   = ax.imshow(disp.T, cmap=cmap_em, vmin=0, vmax=1, aspect='equal', origin='lower')

# Anotaciones
best_h, best_a = np.unravel_index(mat6.argmax(), mat6.shape)
for h in range(N_DISP):
    for a in range(N_DISP):
        p   = mat6[h, a]
        t   = math.sqrt(p / mat6.max())
        color = '#09090b' if t > 0.55 else '#f4f4f5' if t > 0.2 else '#52525b'
        star  = '★\n' if (h == best_h and a == best_a) else ''
        label = f'{star}{p*100:.1f}%' if p >= 0.004 else ''
        ax.text(h, a, label, ha='center', va='center',
                fontsize=8, color=color, fontweight='bold')

# Borde del más probable (amber)
rect = plt.Rectangle((best_h - 0.5, best_a - 0.5), 1, 1,
                      fill=False, edgecolor='#f59e0b', linewidth=2)
ax.add_patch(rect)

ax.set_xticks(range(N_DISP)); ax.set_xticklabels(range(N_DISP))
ax.set_yticks(range(N_DISP)); ax.set_yticklabels(range(N_DISP))
ax.set_xlabel('Goles ARG (local)', color='#a1a1aa')
ax.set_ylabel('Goles FRA (visitante)', color='#a1a1aa')
ax.set_title(f'Heatmap de marcadores — ARG vs FRA\n'
             f'λARG={lam_H_dc:.3f}  λFRA={lam_A_dc:.3f}  [Analítico, sin Elo]',
             color='#e4e4e7', pad=10)

plt.colorbar(im, ax=ax, label='P(h,a) — normalizado a √máx')
plt.tight_layout()
plt.savefig('heatmap_arg_fra.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Marcador más probable: {best_h}–{best_a}  ({mat6[best_h,best_a]*100:.2f}%)')

In [ ]:
# ── Barras 1X2 ───────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(6, 2.2))
ax.set_facecolor('#09090b'); fig.patch.set_facecolor('#09090b')

labels  = [f'ARG (local)',  'Empate (X)',  f'FRA (visit.)']
values  = [p_home_dc, p_draw_dc, p_away_dc]
colors  = ['#10b981', '#71717a', '#8b5cf6']

bars = ax.barh(labels[::-1], [v*100 for v in values[::-1]],
               color=colors[::-1], height=0.55, edgecolor='none')
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', color='#e4e4e7', fontsize=11, fontweight='bold')

ax.set_xlim(0, 60); ax.set_xlabel('Probabilidad (%)', color='#a1a1aa')
ax.set_title('Resultado 1X2 — ARG vs FRA (Analítico)', color='#e4e4e7')
ax.spines[:].set_color('#27272a')
ax.tick_params(colors='#71717a')
plt.tight_layout()
plt.savefig('bars_1x2_arg_fra.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Top-10 marcadores exactos ─────────────────────────────────────────────────

scores = []
for h in range(MAX_GOALS):
    for a in range(MAX_GOALS):
        scores.append({'score': f'{h}-{a}', 'p': mat_dc[h, a], 'h': h, 'a': a})
scores.sort(key=lambda x: -x['p'])

top10 = scores[:10]
max_p = top10[0]['p']

fig, ax = plt.subplots(figsize=(7, 4))
ax.set_facecolor('#09090b'); fig.patch.set_facecolor('#09090b')

for i, s in enumerate(top10):
    h, a, p = s['h'], s['a'], s['p']
    outcome = 'ARG' if h > a else 'FRA' if h < a else 'X'
    color   = '#10b981' if h > a else '#8b5cf6' if h < a else '#71717a'
    bar_w   = (p / max_p) * 5.5
    row     = 9 - i
    ax.barh(row, bar_w, color='#10b981', height=0.6, left=1)
    ax.text(0.05, row, f'{i+1}.', va='center', color='#52525b', fontsize=9)
    ax.text(0.45, row, s['score'], va='center', color='#f4f4f5', fontsize=11,
            fontweight='black', family='monospace')
    ax.text(6.7, row, f'{p*100:.2f}%', va='center', color='#34d399', fontsize=9,
            fontweight='bold', ha='right')
    ax.text(7.0, row, outcome, va='center', color=color, fontsize=9,
            fontweight='bold')

ax.set_xlim(0, 7.5); ax.set_ylim(-0.5, 9.5)
ax.axis('off')
ax.set_title('Top 10 marcadores exactos — ARG vs FRA (Analítico)', color='#e4e4e7')
plt.tight_layout()
plt.savefig('top10_arg_fra.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5:')
for s in top10[:5]:
    h, a = s['h'], s['a']
    out  = 'ARG' if h > a else 'FRA' if h < a else 'X'
    print(f'  {s["score"]}  {s["p"]*100:.2f}%  [{out}]')

## 7 — Simulación Monte Carlo

Python usa `numpy.random.Generator.poisson()`, que internamente usa el método de Marsaglia-Tsang (o Knuth para λ pequeños).  
El JS usa el **algoritmo de Knuth** explícito.

Ambos muestrean de la **misma distribución** `Poisson(λ)` — las secuencias individuales difieren pero las distribuciones empíricas convergen:

```
JS:  k = samplePoisson(lambda, mulberry32(seed))
Py:  k = min(rng.poisson(lambda), 12)
```


In [ ]:
N_SIMS = 50_000
rng_py = np.random.default_rng(FIXED_SEED)

print(f'Simulando {N_SIMS:,} partidos ARG vs FRA (DC+Elo)…')
gh_mc = np.minimum(rng_py.poisson(lam_H_mc, N_SIMS), 12)  # Knuth JS tiene cap en 12
ga_mc = np.minimum(rng_py.poisson(lam_A_mc, N_SIMS), 12)

ph_mc = float(np.mean(gh_mc > ga_mc))
pd_mc = float(np.mean(gh_mc == ga_mc))
pa_mc = float(np.mean(gh_mc < ga_mc))

print(f'\n1X2 MC Python  : {ph_mc*100:.1f}% / {pd_mc*100:.1f}% / {pa_mc*100:.1f}%')
if JS_LOADED:
    js_ph_mc = JS['monteCarlo']['pHome']
    js_pd_mc = JS['monteCarlo']['pDraw']
    js_pa_mc = JS['monteCarlo']['pAway']
    print(f'1X2 MC JS      : {js_ph_mc*100:.1f}% / {js_pd_mc*100:.1f}% / {js_pa_mc*100:.1f}%')
    # Tolerancia 1% (varianza de Monte Carlo con 50k sims)
    assert abs(ph_mc - js_ph_mc) < 0.02, f'pHome MC difiere demasiado: {ph_mc:.4f} vs {js_ph_mc}'
    assert abs(pd_mc - js_pd_mc) < 0.02
    assert abs(pa_mc - js_pa_mc) < 0.02
    print('  ✓ P(1/X/2) MC dentro de ±2% entre Python y JS (varianza esperada)')

In [ ]:
# Top-10 marcadores desde MC Python
from collections import Counter

counts_py = Counter(zip(gh_mc, ga_mc))
top10_mc  = sorted(counts_py.items(), key=lambda x: -x[1])[:10]

print('Top 10 MC Python (DC+Elo):')
for (h, a), cnt in top10_mc:
    out = 'ARG' if h > a else 'FRA' if h < a else 'X'
    print(f'  {h}-{a}  {cnt/N_SIMS*100:.2f}%  [{out}]')

if JS_LOADED:
    print('\nTop 10 MC JS (DC+Elo):')
    for score, cnt in list(JS['monteCarlo']['top30'].items())[:10]:
        print(f'  {score}  {cnt/JS["monteCarlo"]["nSims"]*100:.2f}%')

## 8 — Comparación Analítico vs Monte Carlo

El modo Analítico y MC difieren porque MC incluye el **tilt Elo**:

| Modo | λARG | λFRA | P(ARG gana) |
|------|------|------|-------------|
| Analítico (DC) | `lam_H_dc` | `lam_A_dc` | `p_home_dc` |
| MC (DC+Elo)    | `lam_H_mc` | `lam_A_mc` | `ph_mc`     |


In [ ]:
print('=== RESUMEN DE EQUIVALENCIA ===')
print()
print('Lambdas:')
print(f'  Analítico: λARG={lam_H_dc:.4f}  λFRA={lam_A_dc:.4f}')
print(f'  MC+Elo:    λARG={lam_H_mc:.4f}  λFRA={lam_A_mc:.4f}')
print(f'  Diferencia λARG: {(lam_H_mc-lam_H_dc):.4f} ({(lam_H_mc/lam_H_dc-1)*100:+.2f}%)')
print(f'  Diferencia λFRA: {(lam_A_mc-lam_A_dc):.4f} ({(lam_A_mc/lam_A_dc-1)*100:+.2f}%)')
print()
print('P(1/X/2):')
print(f'  Analítico: {p_home_dc*100:.1f}% / {p_draw_dc*100:.1f}% / {p_away_dc*100:.1f}%')
print(f'  MC Python: {ph_mc*100:.1f}% / {pd_mc*100:.1f}% / {pa_mc*100:.1f}%')
if JS_LOADED:
    print(f'  MC JS:     {JS["monteCarlo"]["pHome"]*100:.1f}% / {JS["monteCarlo"]["pDraw"]*100:.1f}% / {JS["monteCarlo"]["pAway"]*100:.1f}%')
print()
print('El tilt Elo (ARG Elo=2076 > FRA Elo=2003, Δ=73 puntos)')
print('aumenta λARG y reduce λFRA, favoreciendo levemente a Argentina.')
print()
print('Equivalencia matemática:')
print('  scipy.stats.poisson.pmf(k, λ)  ≡  poisson(lambda, k)  en dixonColes.js')
print('  numpy.random.Generator.poisson  ≡  samplePoisson(lambda, rng)  en monteCarlo.js')
print('  (misma distribución Poisson, diferente PRNG — divergencia esperada en MC)')

## Apéndice: Algoritmo de Knuth (Port del sampler JS)

El `samplePoisson` de `monteCarlo.js` usa el algoritmo de Knuth (1969):

```
L = exp(-lambda)
k = 0, p = 1
while p > L: k++, p *= U[0,1)
return min(k-1, 12)
```

Esta celda lo implementa en Python para demostrar que produce la misma **distribución** que `numpy.random.poisson`.


In [ ]:
def sample_poisson_knuth(lam: float, rng_py) -> int:
    """Port exacto de samplePoisson() de monteCarlo.js."""
    L = math.exp(-lam)
    k, p = 0, 1.0
    while p > L:
        k += 1
        p *= rng_py.random()
    return min(k - 1, 12)

# Comparar distribución empírica vs scipy PMF
N_TEST   = 200_000
lam_test = lam_H_mc
rng_cmp  = np.random.default_rng(0)

knuth_samples = np.array([sample_poisson_knuth(lam_test, rng_cmp) for _ in range(N_TEST)])
numpy_samples = np.minimum(np.random.default_rng(1).poisson(lam_test, N_TEST), 12)

k_vals = np.arange(8)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.set_facecolor('#09090b'); fig.patch.set_facecolor('#09090b')

ax.bar(k_vals - 0.25, [np.mean(knuth_samples == k) for k in k_vals], 0.22,
       color='#10b981', label='Knuth (JS port)', alpha=0.85)
ax.bar(k_vals,        [np.mean(numpy_samples  == k) for k in k_vals], 0.22,
       color='#8b5cf6', label='numpy.random.poisson', alpha=0.85)
ax.bar(k_vals + 0.25, stats.poisson.pmf(k_vals, lam_test), 0.22,
       color='#fbbf24', label='scipy PMF (teórico)', alpha=0.85)

ax.set_xlabel('Goles (k)', color='#a1a1aa')
ax.set_ylabel('Probabilidad empírica', color='#a1a1aa')
ax.set_title(f'Equivalencia del sampler Poisson (λ={lam_test:.3f}, N={N_TEST:,})', color='#e4e4e7')
ax.legend(facecolor='#18181b', labelcolor='#e4e4e7')
ax.spines[:].set_color('#27272a')
ax.tick_params(colors='#71717a')
plt.tight_layout()
plt.savefig('sampler_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Kolmogorov-Smirnov: Knuth vs numpy
from scipy.stats import ks_2samp
ks_stat, ks_p = ks_2samp(knuth_samples, numpy_samples)
print(f'Test KS (Knuth vs numpy): estadístico={ks_stat:.4f}  p-value={ks_p:.3f}')
print('p > 0.05 → no se rechaza H₀: ambos muestrean de la misma distribución.')